# REA Colab Quickstart

Run independent, substantial masked Barker SMC replicates from a fresh Colab GPU runtime. Each seed is saved separately, and the final cells compare weighted target statistics across seeds.

In [ ]:
# EDIT PARAMETERS HERE
ALGORITHMS_TO_RUN = [
    "masked_barker_smc",
]

SEEDS = [12345, 12346, 12347]
NUM_PARTICLES = 64
MAX_NEW_TOKENS = 100
LAMBDAS = "0,-0.25,-0.5,-1.0,-1.5,-2.0"
MCMC_STEPS_PER_LEVEL = 5
ESS_THRESHOLD = 0.5
MASK_MODE = "stochastic"
MASK_SIZES = "1,2,4,8"
MASK_RHO_SCHEDULE = "0.4,0.3,0.2,0.1"
MASK_POLICY = "uniform_noncontiguous"
PROPOSAL_TEMPERATURE = 1.0
BATCH_SIZE = 32
EXPERIMENT_LABEL = "masked_barker_smc_multiseed"
SKIP_COMPLETED_RUNS = True

VALID_ALGORITHMS = {"masked_barker_smc"}
unknown = set(ALGORITHMS_TO_RUN) - VALID_ALGORITHMS
if unknown:
    raise ValueError(f"Unknown algorithms: {sorted(unknown)}")
if not SEEDS or len(set(SEEDS)) != len(SEEDS) or not all(isinstance(seed, int) for seed in SEEDS):
    raise ValueError("SEEDS must contain distinct integers")
print("Selected:", ALGORITHMS_TO_RUN)
print("Seeds:", SEEDS)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import pathlib
import subprocess

REPO_URL = "https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main.git"
REPO_DIR = pathlib.Path("/content/rea_of_llms_supplementary_code-main")
OUTPUT_ROOT = pathlib.Path("/content/drive/MyDrive/rea_outputs")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("cwd =", os.getcwd())
print("outputs =", OUTPUT_ROOT)

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
import torch
print("torch:", torch.__version__)
print("compiled CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Run selected pipelines

Each seed writes to its own Drive directory. Completed runs can be skipped safely, and no other sampler is imported or executed.

In [ ]:
import json

experiment_root = OUTPUT_ROOT / EXPERIMENT_LABEL
experiment_root.mkdir(parents=True, exist_ok=True)
run_dirs = {}
expected_lambdas = [float(value) for value in LAMBDAS.split(",")]
expected_mask_sizes = [int(value) for value in MASK_SIZES.split(",")]
expected_rho_schedule = [
    [float(value) for value in row.split(",")]
    for row in MASK_RHO_SCHEDULE.split(";")
]

def expected_run_config(seed):
    return {
        "seed": seed,
        "num_particles": NUM_PARTICLES,
        "max_new_tokens": MAX_NEW_TOKENS,
        "lambdas": expected_lambdas,
        "mcmc_steps_per_level": MCMC_STEPS_PER_LEVEL,
        "ess_threshold": ESS_THRESHOLD,
        "resampling": True,
        "resampling_method": "systematic",
        "mask_mode": MASK_MODE,
        "mask_sizes": expected_mask_sizes,
        "mask_rho_schedule": expected_rho_schedule,
        "mask_policy": MASK_POLICY,
        "proposal_temperature": PROPOSAL_TEMPERATURE,
        "batch_size": BATCH_SIZE,
    }

def run_config_mismatches(payload, seed):
    if payload.get("algorithm") != "masked_barker_smc":
        return [f"algorithm: saved={payload.get('algorithm')!r}, expected='masked_barker_smc'"]
    saved = payload.get("config", {})
    return [
        f"{key}: saved={saved.get(key)!r}, expected={expected!r}"
        for key, expected in expected_run_config(seed).items()
        if saved.get(key) != expected
    ]

def masked_barker_command(seed, output_dir):
    run_name = f"{EXPERIMENT_LABEL}_seed_{seed}"
    return [
        "python", "masked_barker_smc_experiment/run_experiment.py",
        "--device", DEVICE,
        "--num-particles", str(NUM_PARTICLES),
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--lambdas", LAMBDAS,
        "--mcmc-steps-per-level", str(MCMC_STEPS_PER_LEVEL),
        "--ess-threshold", str(ESS_THRESHOLD),
        "--mask-mode", MASK_MODE,
        "--mask-sizes", MASK_SIZES,
        "--mask-rho-schedule", MASK_RHO_SCHEDULE,
        "--mask-policy", MASK_POLICY,
        "--proposal-temperature", str(PROPOSAL_TEMPERATURE),
        "--batch-size", str(BATCH_SIZE),
        "--seed", str(seed),
        "--run-name", run_name,
        "--output-dir", str(output_dir),
    ]

for algorithm in ALGORITHMS_TO_RUN:
    if algorithm != "masked_barker_smc":
        raise ValueError(f"Unsupported algorithm in this notebook: {algorithm}")
    for seed in SEEDS:
        output_dir = experiment_root / f"seed_{seed}"
        run_dirs[(algorithm, seed)] = output_dir
        summary_path = output_dir / "summary.json"
        samples_path = output_dir / "samples.jsonl"
        complete = summary_path.exists() and samples_path.exists()
        mismatches = []
        if complete:
            try:
                saved_payload = json.loads(summary_path.read_text(encoding="utf-8"))
                mismatches = run_config_mismatches(saved_payload, seed)
            except (json.JSONDecodeError, OSError, KeyError, TypeError) as error:
                mismatches = [f"unreadable summary: {error}"]
        if SKIP_COMPLETED_RUNS and complete and not mismatches:
            print(f"Skipping matching completed run: {algorithm}, seed={seed}")
            continue
        if complete and mismatches:
            print(f"Saved run for seed {seed} has a different configuration; rerunning.")
            for mismatch in mismatches:
                print("  -", mismatch)
        print(f"\nRunning {algorithm}, seed={seed} ...", flush=True)
        subprocess.run(masked_barker_command(seed, output_dir), check=True)

In [ ]:
from IPython.display import Markdown, display

for seed in SEEDS:
    run_dir = run_dirs[("masked_barker_smc", seed)]
    report_path = run_dir / "report.md"
    summary_path = run_dir / "summary.json"
    if not report_path.exists():
        print(f"No report found for seed {seed}: {report_path}")
        continue
    display(Markdown(f"## Seed {seed}"))
    display(Markdown(report_path.read_text(encoding="utf-8")))
    print("Summary:", summary_path)

## Validate and compare the multi-seed runs

The next cells verify every particle output, distinguish weighted target statistics from raw particle summaries, save aggregate CSV files, and compare diagnostics across seeds. They do not launch another simulation.

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display

if "masked_barker_smc" not in ALGORITHMS_TO_RUN:
    raise RuntimeError("Add 'masked_barker_smc' to ALGORITHMS_TO_RUN and execute the run cell first.")

run_rows = []
level_frames = []
sample_frames = []
expected_lambdas = [float(value) for value in LAMBDAS.split(",")]

for seed in SEEDS:
    run_dir = run_dirs[("masked_barker_smc", seed)]
    summary_path = run_dir / "summary.json"
    samples_path = run_dir / "samples.jsonl"
    if not summary_path.exists() or not samples_path.exists():
        raise FileNotFoundError(f"Missing completed output for seed {seed}; execute the run cell first.")

    payload = json.loads(summary_path.read_text(encoding="utf-8"))
    seed_samples = pd.read_json(samples_path, lines=True)
    seed_levels = pd.DataFrame(payload["diagnostics"]["level_diagnostics"])
    weights = seed_samples["normalized_weight"].to_numpy(dtype=float)
    raw_ari = seed_samples["raw_ari"].to_numpy(dtype=float)

    mismatches = run_config_mismatches(payload, seed)
    if mismatches:
        details = "\n".join(f"  - {item}" for item in mismatches)
        raise ValueError(
            f"Saved output for seed {seed} does not match the current configuration:\n{details}\n"
            "Rerun the simulation cell; it will now detect and replace stale outputs."
        )
    assert len(seed_samples) == NUM_PARTICLES
    assert seed_samples["completion_token_ids"].map(len).eq(MAX_NEW_TOKENS).all()
    assert np.isfinite(weights).all() and np.isclose(weights.sum(), 1.0, atol=1e-6)
    assert np.isfinite(raw_ari).all()
    assert len(seed_levels) == len(expected_lambdas)
    assert 0.0 < payload["final_ess_over_n"] <= 1.0
    assert np.isclose(payload["unweighted_mean_ari"], raw_ari.mean())
    assert np.isclose(payload["weighted_mean_ari"], np.sum(weights * raw_ari))

    run_row = {
        "seed": seed,
        "unweighted_mean_ari": payload["unweighted_mean_ari"],
        "weighted_mean_ari": payload["weighted_mean_ari"],
        "min_ari": payload["min_ari"],
        "max_ari": payload["max_ari"],
        "final_ess_over_n": payload["final_ess_over_n"],
        "min_ess_over_n": payload["min_ess_over_n"],
        "acceptance_rate": payload["mean_mcmc_acceptance_rate"],
        "resampling_steps": payload["resampling_steps"],
        "runtime_seconds": payload["runtime_total_seconds"],
    }
    for key, count in payload["tail_counts"].items():
        suffix = key.replace("ari_ge_", "ge_", 1)
        run_row[f"raw_count_{suffix}"] = count
        run_row[f"raw_proportion_{suffix}"] = payload["tail_proportions"][key]
        run_row[f"weighted_probability_{suffix}"] = payload["weighted_tail_probabilities"][key]
    run_rows.append(run_row)

    seed_levels.insert(0, "seed", seed)
    seed_samples.insert(0, "seed", seed)
    level_frames.append(seed_levels)
    sample_frames.append(seed_samples)

runs_df = pd.DataFrame(run_rows)
levels_df = pd.concat(level_frames, ignore_index=True)
samples_df = pd.concat(sample_frames, ignore_index=True)
runs_df.to_csv(experiment_root / "multi_seed_summary.csv", index=False)
levels_df.to_csv(experiment_root / "multi_seed_level_diagnostics.csv", index=False)

print("All multi-seed end-to-end checks passed.")
display(runs_df)

aggregate_columns = [
    "unweighted_mean_ari", "weighted_mean_ari", "max_ari",
    "final_ess_over_n", "min_ess_over_n", "acceptance_rate", "runtime_seconds",
]
aggregate_columns += [
    column for column in runs_df.columns
    if column.startswith(("raw_proportion_", "weighted_probability_"))
]
aggregate_df = runs_df[aggregate_columns].agg(["mean", "std"]).T.reset_index()
aggregate_df.columns = ["metric", "mean_across_seeds", "std_across_seeds"]
display(aggregate_df)
print("Saved:", experiment_root / "multi_seed_summary.csv")
print("Saved:", experiment_root / "multi_seed_level_diagnostics.csv")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for seed, seed_levels in levels_df.groupby("seed"):
    axes[0, 0].plot(
        seed_levels["lambda"], seed_levels["weighted_mean_ari_before_resampling"],
        marker="o", label=f"weighted pre, seed {seed}",
    )
    axes[0, 0].plot(
        seed_levels["lambda"], seed_levels["unweighted_mean_ari_after_resampling"],
        marker="x", linestyle="--", label=f"unweighted post, seed {seed}",
    )
axes[0, 0].set(title="ARI before and after resampling", xlabel="lambda", ylabel="raw ARI")
axes[0, 0].legend(fontsize=8, ncol=2)
axes[0, 0].grid(alpha=0.25)

for seed, seed_levels in levels_df.groupby("seed"):
    axes[0, 1].plot(
        seed_levels["lambda"], seed_levels["ess_over_n_before_resampling"],
        marker="o", label=f"seed {seed}",
    )
axes[0, 1].axhline(ESS_THRESHOLD, color="tab:red", linestyle="--", label="ESS threshold")
axes[0, 1].set(title="Pre-resampling particle degeneracy", xlabel="lambda", ylabel="ESS / N", ylim=(0, 1.05))
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(alpha=0.25)

for seed, seed_levels in levels_df[levels_df["level"] > 0].groupby("seed"):
    axes[1, 0].plot(
        seed_levels["lambda"], seed_levels["acceptance_rate"],
        marker="o", label=f"seed {seed}",
    )
axes[1, 0].set(title="Mutation acceptance", xlabel="new lambda after weight update", ylabel="acceptance rate", ylim=(0, 1.05))
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.25)

grouped_ari = [
    samples_df.loc[samples_df["seed"] == seed, "raw_ari"].to_numpy(dtype=float)
    for seed in SEEDS
]
axes[1, 1].boxplot(grouped_ari, tick_labels=[str(seed) for seed in SEEDS], showmeans=True)
axes[1, 1].set(title="Final raw ARI by seed", xlabel="seed", ylabel="raw ARI")
axes[1, 1].grid(axis="y", alpha=0.25)

fig.tight_layout()
plt.show()

In [ ]:
top_completions = samples_df.sort_values("raw_ari", ascending=False).drop_duplicates("completion").head(10)
display(top_completions[["seed", "particle_id", "raw_ari", "capped_ari", "normalized_weight", "completion"]])

## Optional selected-result publication

A normal run writes only to Drive. To add a reviewed run to the repository, rerun its command with `--experiment-log results/EXPERIMENT_LOG.md --publish-result-dir results/selected_runs`, inspect the small generated files, then commit them intentionally.